# Track B — B1.4 no Google Colab

Abra **Ambiente de execução → Alterar tipo de ambiente de execução → GPU** (T4, se disponível).
Execute as células em ordem. Quando solicitado, envie **track_b_colab_package_v1.zip**.
O notebook faz um teste de 10 passos; o treino completo começa somente na célula própria.
As sementes 7, 42 e 2026 usam exatamente as divisões congeladas dos baselines.
O teste final DSL-TL não faz parte deste pacote.

O Drive é opcional, mas recomendado para sobreviver ao encerramento da sessão.
Se usado, o Colab solicitará sua autorização. Reserve espaço para pesos e checkpoints
(aproximadamente 10 GB para as três sementes, dependendo dos artefatos).
Os escores do teste curto não serão usados para selecionar o modelo.


In [ ]:
from pathlib import Path
import hashlib, io, json, subprocess, sys, zipfile
from google.colab import files

uploaded = files.upload()
zip_names = [filename for filename in uploaded if filename.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Envie apenas o ZIP do pacote nesta célula.')
name = zip_names[0]
EXTRACT_ROOT = Path('/content/pt_variant_b1_4')
EXTRACT_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    for item in archive.infolist():
        destination = (EXTRACT_ROOT / item.filename).resolve()
        if not destination.is_relative_to(EXTRACT_ROOT.resolve()):
            raise ValueError('Caminho inválido dentro do pacote.')
    archive.extractall(EXTRACT_ROOT)
PACKAGE_ROOT = EXTRACT_ROOT / 'track_b_colab'
manifest = json.loads((PACKAGE_ROOT / 'package_manifest.json').read_text())
for filename, expected in manifest['sha256'].items():
    actual = hashlib.sha256((PACKAGE_ROOT / filename).read_bytes()).hexdigest()
    if actual != expected:
        raise ValueError(f'Hash divergente: {filename}')
print('Pacote validado:', manifest['rows'], 'textos; sementes', manifest['seeds'])


## Dependências
O ambiente manterá o PyTorch/CUDA que o Colab fornece. O treino roda em um processo
Python novo, para usar as versões instaladas aqui. O BERTimbau público será baixado
na primeira execução; nenhuma chave de API é necessária.


In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(PACKAGE_ROOT / 'requirements-colab.txt')
], check=True)
subprocess.run([sys.executable, '-c',
    "import torch; assert torch.cuda.is_available(), 'GPU ausente: selecione GPU e reconecte'; "
    "print('GPU:', torch.cuda.get_device_name(0)); "
    "print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)"
], check=True)


## Persistência
Mantenha USE_DRIVE=True para salvar checkpoints no Drive. A pasta é exclusiva deste
experimento. Se uma sessão cair, reabra este notebook, selecione GPU, execute as
células anteriores e repita a célula de treino: as sementes concluídas serão
reconhecidas e a semente incompleta retomará o último checkpoint.
Com USE_DRIVE=False, os arquivos em /content podem desaparecer quando a sessão terminar.


In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/pt-variant-track-b/b1_4_colab_v1')
else:
    OUTPUT_ROOT = Path('/content/b1_4_colab_v1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Resultados e checkpoints:', OUTPUT_ROOT)


## Teste curto: tempo e memória
Execute primeiro. Ele testa download do modelo, treinamento FP16, validação e gravação
de checkpoint. É separado do experimento completo. A estimativa usa a taxa observada,
mas escrita no Drive e interrupções podem alterar o tempo final.


In [ ]:
def run_training(mode, seed=None):
    command = [sys.executable, str(PACKAGE_ROOT / 'train_b1_4.py'),
               '--package-root', str(PACKAGE_ROOT), '--output-dir', str(OUTPUT_ROOT),
               '--mode', mode]
    if seed is not None:
        command += ['--seed', str(seed)]
    subprocess.run(command, check=True)

run_training('smoke')
smoke = json.loads((OUTPUT_ROOT / 'smoke/seed_42/report.json').read_text())
print('GPU:', smoke['gpu'])
print('Pico de memória GPU:', round(smoke['peak_allocated_gpu_gib'], 2), 'GiB')
seconds_per_step = smoke['training_metrics']['train_runtime'] / smoke['global_steps']
rough_minutes = seconds_per_step * smoke['planned_full_steps_per_seed'] * 3 / 60
print('Estimativa inicial das três sementes:', round(rough_minutes, 1), 'minutos')
print('Teste curto concluído. A célula abaixo inicia o experimento completo.')


## Treinamento completo
Esta célula executa as três sementes em sequência. São três épocas por semente,
learning rate 2e-5, batch 8 com acumulação de 2, máximo de 256 tokens.
São avaliados e salvos checkpoints a cada 50 passos; o melhor F1 macro de validação
define o checkpoint escolhido dentro de cada execução. Isto não é treino final
para publicação: o resultado será comparado aos outros baselines locais.
Para rodar apenas uma semente, use por exemplo run_training('full', seed=7).


In [ ]:
run_training('full')
aggregate_path = OUTPUT_ROOT / 'full/aggregate.json'
if aggregate_path.exists():
    aggregate = json.loads(aggregate_path.read_text())
    print('F1 macro médio:', round(aggregate['macro_f1_mean'], 6))
    print('Desvio entre sementes:', round(aggregate['macro_f1_std'], 6))


## Baixar resultados para análise aqui
A exportação pequena contém relatórios, predições e configurações; não inclui
checkpoints pesados. Pesos e checkpoints continuam na pasta de resultados.
Você pode baixar esses modelos pelo Drive depois de compararmos os resultados.


In [ ]:
results_zip = Path('/content/track_b_b1_4_results_v1.zip')
with zipfile.ZipFile(results_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_ROOT.rglob('*')):
        if path.is_file() and path.name in {'report.json', 'predictions.parquet', 'run_config.json', 'aggregate.json'}:
            archive.write(path, arcname=path.relative_to(OUTPUT_ROOT))
files.download(str(results_zip))
print('Guarde o ZIP e copie-o depois para a pasta runs/track_b do projeto local.')
